In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import nnls


# File path setup
path_cwd = Path.cwd()
path_input = str(path_cwd) + '/Input_files/'
path_output_graphs = str(path_cwd) + '/Output_graphs/'
path_graphs=path_output_graphs+'Graphs/'

colum_names=['Date','Type','ID','Location','Soil depth (cm)','Delta 2H (‰)','Delta 18O (‰)']

df_2021 = pd.read_csv(path_input + 'isotopes_2021.csv', header=None, skiprows=1,names=colum_names)
df_2022 = pd.read_csv(path_input + 'isotopes_2022.csv', header=None, skiprows=1,names=colum_names)
# Convert Date to datetime for consistency
df_2021['Date'] = pd.to_datetime(df_2021['Date'])
df_2022['Date'] = pd.to_datetime(df_2022['Date'])



In [2]:
# Calculate lc-excess for each sample using the regional LMWL: δ²H = 7.48 × δ¹⁸O - 3.70
df_2021['lc-excess'] = df_2021['Delta 2H (‰)'] - (7.48 * df_2021['Delta 18O (‰)'] + 3.70)
df_2022['lc-excess'] = df_2022['Delta 2H (‰)'] - (7.48 * df_2022['Delta 18O (‰)'] + 3.70)

# Combine dataframes for analysis
df_all = pd.concat([df_2021, df_2022], ignore_index=True)

# Extract year and month for grouping
df_all['Year'] = df_all['Date'].dt.year
df_all['Month'] = df_all['Date'].dt.strftime('%m')

# Group by Year, Month, and Type, calculate mean and std of lc-excess
lc_excess_stats = df_all.groupby(['Year', 'Month', 'Type'])['lc-excess'].agg(['mean', 'std']).reset_index()

# Create separate figures for each year
for year in [2021, 2022]:
    year_data = lc_excess_stats[lc_excess_stats['Year'] == year]
    
    # Exclude October (month '10') for 2022
    if year == 2022:
        year_data = year_data[year_data['Month'] != '10']
    
    # Pivot data for plotting (columns: types, index: months)
    pivot_mean = year_data.pivot(index='Month', columns='Type', values='mean')
    pivot_std = year_data.pivot(index='Month', columns='Type', values='std').fillna(0)  # Fill NaN std with 0 for single samples
    
    # Create a Plotly figure for the current year
    fig = go.Figure()
    
    # Add traces for each type with markers and error bars, using specified colors
    if 'Water' in pivot_mean.columns:
        fig.add_trace(go.Scatter(
            x=pivot_mean.index, 
            y=pivot_mean['Water'], 
            mode='markers', 
            name='Precipitation',
            error_y=dict(type='data', array=pivot_std['Water'], visible=True),
            marker_color='cornflowerblue'
        ))
    if 'Fir' in pivot_mean.columns:
        fig.add_trace(go.Scatter(
            x=pivot_mean.index, 
            y=pivot_mean['Fir'], 
            mode='markers', 
            name='Fir',
            error_y=dict(type='data', array=pivot_std['Fir'], visible=True),
            marker_color='palegreen'
        ))
    if 'Spruce' in pivot_mean.columns:
        fig.add_trace(go.Scatter(
            x=pivot_mean.index, 
            y=pivot_mean['Spruce'], 
            mode='markers', 
            name='Spruce',
            error_y=dict(type='data', array=pivot_std['Spruce'], visible=True),
            marker_color='darkgreen'
        ))
    if 'Soil' in pivot_mean.columns:
        fig.add_trace(go.Scatter(
            x=pivot_mean.index, 
            y=pivot_mean['Soil'], 
            mode='markers', 
            name='Soil',
            error_y=dict(type='data', array=pivot_std['Soil'], visible=True),
            marker_color='#F4A460'  # Apricot orange approximation
        ))
    
    # Update layout for the current year with month names
    fig.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=['05', '06', '07', '08', '09'],
            ticktext=['May', 'June', 'July', 'August', 'September']
        ),
        xaxis_title='Month',
        yaxis_title='lc-excess (‰)',
        legend_title='Type',
        height=500,
        width=800
    )
    
    # Show plot for the current year
    fig.show()
    path_isotopes = path_graphs + 'Isotopes/'
    fig.write_image(path_isotopes + 'ic-excess_' + str(year) + '.png', scale=8)
    fig.write_html(path_isotopes + 'ic-excess_' + str(year) + '.html', include_plotlyjs='cdn')

In [4]:
colum_names = ['Date', 'Type', 'ID', 'Location', 'Soil depth (cm)', 'Delta 2H (‰)', 'Delta 18O (‰)']

# Read and process 2021 data
df_2021 = pd.read_csv(path_input + 'isotopes_2021.csv', header=None, skiprows=1, names=colum_names)
df_2021['Date'] = pd.to_datetime(df_2021['Date'])
df_2021['Month'] = df_2021['Date'].dt.strftime('%m')
isotope_stats_2021 = df_2021.groupby(['Month', 'Type'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index()
isotope_stats_2021['Year'] = 2021

# Read and process 2022 data with soil depth split
df_2022 = pd.read_csv(path_input + 'isotopes_2022.csv', header=None, skiprows=1, names=colum_names)
df_2022['Date'] = pd.to_datetime(df_2022['Date'])
df_2022['Month'] = df_2022['Date'].dt.strftime('%m')
df_2022_with_depth = df_2022.dropna(subset=['Soil depth (cm)'])
df_2022_with_depth['Soil Depth Category'] = np.where(df_2022_with_depth['Soil depth (cm)'] < 3, 'Surface', 'Deep')
isotope_stats_2022_base = df_2022.groupby(['Month', 'Type'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index()
isotope_stats_2022_depth = df_2022_with_depth.groupby(['Month', 'Soil Depth Category'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index().rename(columns={'Soil Depth Category': 'Type'})
isotope_stats_2022 = pd.concat([isotope_stats_2022_base, isotope_stats_2022_depth])
isotope_stats_2022['Year'] = 2022

# Function to perform three-endmember mixing with proper NNLS
def three_endmember_mixing(tree_data, precip_current, precip_prev, soil_data):
    # Construct the A matrix (2x3) with isotopic equations only
    A = np.array([
        [precip_current['Delta 2H (‰)'], precip_prev['Delta 2H (‰)'], soil_data['Delta 2H (‰)']],
        [precip_current['Delta 18O (‰)'], precip_prev['Delta 18O (‰)'], soil_data['Delta 18O (‰)']]
    ])
    b = np.array([tree_data['Delta 2H (‰)'], tree_data['Delta 18O (‰)']])
    
    # Solve with NNLS
    fractions, _ = nnls(A, b)
    
    # Normalize to sum to 1, handling zero sums or negative values
    total = fractions.sum()
    if total == 0 or any(f < 0 for f in fractions):
        return [0.0, 0.0, 0.0]  # Fallback if invalid
    f1, f2, f3 = fractions / total
    return f1, f2, f3  # f1: current precip, f2: previous precip, f3: soil

# Analyze each year separately
results = []
for year, year_stats in [(2021, isotope_stats_2021), (2022, isotope_stats_2022)]:
    months = year_stats['Month'].unique()
    month_names = { '06': 'June', '07': 'July', '08': 'August', '09': 'September' }
    for month in months:
        current_month_data = year_stats[year_stats['Month'] == month]
        prev_month = str(int(month) - 1).zfill(2)
        prev_month_data = year_stats[year_stats['Month'] == prev_month] if prev_month in months else None

        if (len(current_month_data[current_month_data['Type'] == 'Water']) > 0 and
            len(current_month_data[current_month_data['Type'].isin(['Fir', 'Spruce'])]) > 0 and
            len(current_month_data[current_month_data['Type'].isin(['Soil', 'Surface', 'Deep'])]) > 0 and
            (prev_month_data is not None and len(prev_month_data[prev_month_data['Type'] == 'Water']) > 0)):
            precip_current = current_month_data[current_month_data['Type'] == 'Water'].iloc[0]
            precip_prev = prev_month_data[prev_month_data['Type'] == 'Water'].iloc[0]
            soil_data = current_month_data[current_month_data['Type'].isin(['Soil', 'Surface', 'Deep'])].iloc[0]
            for _, tree_row in current_month_data[current_month_data['Type'].isin(['Fir', 'Spruce'])].iterrows():
                f1, f2, f3 = three_endmember_mixing(tree_row, precip_current, precip_prev, soil_data)
                # Residuals
                pred_d2h = f1 * precip_current['Delta 2H (‰)'] + f2 * precip_prev['Delta 2H (‰)'] + f3 * soil_data['Delta 2H (‰)']
                pred_d18o = f1 * precip_current['Delta 18O (‰)'] + f2 * precip_prev['Delta 18O (‰)'] + f3 * soil_data['Delta 18O (‰)']
                residual_d2h = abs(tree_row['Delta 2H (‰)'] - pred_d2h)
                residual_d18o = abs(tree_row['Delta 18O (‰)'] - pred_d18o)
                results.append({
                    'Year': year,
                    'Month': month_names[month],  # Changed to month names
                    'Type': tree_row['Type'],
                    'Fraction_Current_Precip': f1,
                    'Fraction_Prev_Precip': f2,
                    'Fraction_Surface_Soil': f3,  # Changed from Fraction_Soil
                    'Residual_D2H': residual_d2h,
                    'Residual_D18O': residual_d18o,
                    'Delta_2H_Tree': tree_row['Delta 2H (‰)'],
                    'Delta_18O_Tree': tree_row['Delta 18O (‰)']
                })

results_df = pd.DataFrame(results)

# Separate stacked bar plots for 2021 and 2022 with Fir and Spruce side by side
for year in [2021, 2022]:
    year_data = results_df[results_df['Year'] == year]
    if not year_data.empty:
        fig = go.Figure()

        # Create x-axis labels pairing month names with Fir and Spruce
        months = year_data['Month'].unique()
        x_labels = [f'{month} (Fir)' for month in months] + [f'{month} (Spruce)' for month in months]
        
        # Add traces for each source

        #orange (#E69F00) for surface soil (warm tone for enriched), teal (#56B4E9) for previous precipitation (mid-tone), and navy (#0072B2) for current precipitation (cool, distinct)
        for source, color in [('Fraction_Current_Precip', '#0072B2'),#'cornflowerblue'), 
                             ('Fraction_Prev_Precip', '#56B4E9'),#'lightblue'), 
                             ('Fraction_Surface_Soil', '#FFA500')]:  # Changed to orange
            fir_data = year_data[year_data['Type'] == 'Fir'][source] * 100
            spruce_data = year_data[year_data['Type'] == 'Spruce'][source] * 100
            y_values = list(fir_data) + list(spruce_data)
            fig.add_trace(go.Bar(
                x=x_labels,
                y=y_values,
                name=source.replace('Fraction_', ''),
                marker_color=color
            ))

        fig.update_layout(
            barmode='stack',
            #title=f'Percentage Contribution of Water Sources to Tree Water - {year}',
            xaxis_title='Month and Tree Type',
            yaxis_title='Percentage (%)',
            legend_title='Source',
            height=500,
            width=800,
            xaxis=dict(
                tickmode='array',
                tickvals=[f'{m} (Fir)' for m in months] + [f'{m} (Spruce)' for m in months],
                ticktext=[f'{m} (Fir)' for m in months] + [f'{m} (Spruce)' for m in months]
            )
        )
        fig.show()
        path_isotopes = path_graphs + 'Isotopes/'
        fig.write_image(path_isotopes + 'Percentages_mixing_{year}.png'.format(year=year), scale=5)
        fig.write_html(path_isotopes + 'Percentages_mixing_{year}.html'.format(year=year), include_plotlyjs='cdn')



# Print results for inspection
print("Mixing Model Results:")
print(results_df)

/var/folders/73/1hl1pq915rg4htnkk3r29mym0000gn/T/ipykernel_4050/3571680420.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Mixing Model Results:
    Year      Month    Type  Fraction_Current_Precip  Fraction_Prev_Precip  \
0   2021       June     Fir                 0.179386              0.820614   
1   2021       June  Spruce                 0.000000              0.859658   
2   2021       July     Fir                 0.000000              0.905528   
3   2021       July  Spruce                 0.000000              0.858406   
4   2021     August     Fir                 0.907429              0.092571   
5   2021     August  Spruce                 0.986817              0.000000   
6   2022       June     Fir                 0.000000              0.000000   
7   2022       June  Spruce                 0.000000              0.000000   
8   2022       July     Fir                 0.000000              0.834811   
9   2022       July  Spruce                 0.000000              0.666078   
10  2022     August     Fir                 0.000000              0.399030   
11  2022     August  Spruce               

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import nnls

colum_names = ['Date', 'Type', 'ID', 'Location', 'Soil depth (cm)', 'Delta 2H (‰)', 'Delta 18O (‰)']

# Read and process 2021 data
df_2021 = pd.read_csv(path_input + 'isotopes_2021.csv', header=None, skiprows=1, names=colum_names)
df_2021['Date'] = pd.to_datetime(df_2021['Date'])
df_2021['Month'] = df_2021['Date'].dt.strftime('%m')
isotope_stats_2021 = df_2021.groupby(['Month', 'Type'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index()
isotope_stats_2021['Year'] = 2021

# Read and process 2022 data with soil depth split
df_2022 = pd.read_csv(path_input + 'isotopes_2022.csv', header=None, skiprows=1, names=colum_names)
df_2022['Date'] = pd.to_datetime(df_2022['Date'])
df_2022['Month'] = df_2022['Date'].dt.strftime('%m')
df_2022_with_depth = df_2022.dropna(subset=['Soil depth (cm)'])
df_2022_with_depth['Soil Depth Category'] = np.where(df_2022_with_depth['Soil depth (cm)'] < 3, 'Surface', 'Deep')
isotope_stats_2022_base = df_2022.groupby(['Month', 'Type'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index()
isotope_stats_2022_depth = df_2022_with_depth.groupby(['Month', 'Soil Depth Category'])[['Delta 2H (‰)', 'Delta 18O (‰)']].mean().reset_index().rename(columns={'Soil Depth Category': 'Type'})
isotope_stats_2022 = pd.concat([isotope_stats_2022_base, isotope_stats_2022_depth])
isotope_stats_2022['Year'] = 2022

# Function to perform three-endmember mixing with proper NNLS
def three_endmember_mixing(tree_data, precip_current, precip_prev, soil_data, perturb=0.0):
    # Perturb isotopic values by a percentage (e.g., ±5%)
    perturb_d2h = 1 + (np.random.uniform(-0.05, 0.05) * perturb)
    perturb_d18o = 1 + (np.random.uniform(-0.05, 0.05) * perturb)
    
    A = np.array([
        [precip_current['Delta 2H (‰)'] * perturb_d2h, precip_prev['Delta 2H (‰)'] * perturb_d2h, soil_data['Delta 2H (‰)'] * perturb_d2h],
        [precip_current['Delta 18O (‰)'] * perturb_d18o, precip_prev['Delta 18O (‰)'] * perturb_d18o, soil_data['Delta 18O (‰)'] * perturb_d18o]
    ])
    b = np.array([tree_data['Delta 2H (‰)'], tree_data['Delta 18O (‰)']])
    
    # Solve with NNLS
    fractions, _ = nnls(A, b)
    
    # Normalize to sum to 1, handling zero sums or negative values
    total = fractions.sum()
    if total == 0 or any(f < 0 for f in fractions):
        return [0.0, 0.0, 0.0]
    f1, f2, f3 = fractions / total
    return f1, f2, f3  # f1: current precip, f2: previous precip, f3: soil

# Function to run sensitivity analysis
def sensitivity_analysis(tree_data, precip_current, precip_prev, soil_data, n_simulations=100):
    fractions_list = []
    for _ in range(n_simulations):
        f1, f2, f3 = three_endmember_mixing(tree_data, precip_current, precip_prev, soil_data, perturb=1.0)
        fractions_list.append([f1, f2, f3])
    fractions_array = np.array(fractions_list)
    return np.min(fractions_array, axis=0), np.max(fractions_array, axis=0)

# Analyze each year separately
results = []
for year, year_stats in [(2021, isotope_stats_2021), (2022, isotope_stats_2022)]:
    months = year_stats['Month'].unique()
    month_names = { '06': 'June', '07': 'July', '08': 'August', '09': 'September' }
    for month in months:
        current_month_data = year_stats[year_stats['Month'] == month]
        prev_month = str(int(month) - 1).zfill(2)
        prev_month_data = year_stats[year_stats['Month'] == prev_month] if prev_month in months else None

        if (len(current_month_data[current_month_data['Type'] == 'Water']) > 0 and
            len(current_month_data[current_month_data['Type'].isin(['Fir', 'Spruce'])]) > 0 and
            len(current_month_data[current_month_data['Type'].isin(['Soil', 'Surface', 'Deep'])]) > 0 and
            (prev_month_data is not None and len(prev_month_data[prev_month_data['Type'] == 'Water']) > 0)):
            precip_current = current_month_data[current_month_data['Type'] == 'Water'].iloc[0]
            precip_prev = prev_month_data[prev_month_data['Type'] == 'Water'].iloc[0]
            soil_data = current_month_data[current_month_data['Type'].isin(['Soil', 'Surface', 'Deep'])].iloc[0]
            for _, tree_row in current_month_data[current_month_data['Type'].isin(['Fir', 'Spruce'])].iterrows():
                f1, f2, f3 = three_endmember_mixing(tree_row, precip_current, precip_prev, soil_data, perturb=0.0)
                min_fractions, max_fractions = sensitivity_analysis(tree_row, precip_current, precip_prev, soil_data)
                results.append({
                    'Year': year,
                    'Month': month_names[month],
                    'Type': tree_row['Type'],
                    'Fraction_Current_Precip': f1,
                    'Fraction_Prev_Precip': f2,
                    'Fraction_Surface_Soil': f3,
                    'Min_Current_Precip': min_fractions[0],
                    'Max_Current_Precip': max_fractions[0],
                    'Min_Prev_Precip': min_fractions[1],
                    'Max_Prev_Precip': max_fractions[1],
                    'Min_Surface_Soil': min_fractions[2],
                    'Max_Surface_Soil': max_fractions[2],
                    'Delta_2H_Tree': tree_row['Delta 2H (‰)'],
                    'Delta_18O_Tree': tree_row['Delta 18O (‰)']
                })

results_df = pd.DataFrame(results)

# Separate stacked bar plots for 2021 and 2022 with Fir and Spruce side by side
for year in [2021, 2022]:
    year_data = results_df[results_df['Year'] == year]
    if not year_data.empty:
        fig = go.Figure()

        # Create x-axis labels pairing month names with Fir and Spruce
        months = year_data['Month'].unique()
        x_labels = [f'{month} (Fir)' for month in months] + [f'{month} (Spruce)' for month in months]
        
        # Add traces for each source
        for source, color in [('Fraction_Current_Precip', 'cornflowerblue'), 
                             ('Fraction_Prev_Precip', 'lightblue'), 
                             ('Fraction_Surface_Soil', '#FFA500')]:
            fir_data = year_data[year_data['Type'] == 'Fir'][source] * 100
            spruce_data = year_data[year_data['Type'] == 'Spruce'][source] * 100
            y_values = list(fir_data) + list(spruce_data)
            fig.add_trace(go.Bar(
                x=x_labels,
                y=y_values,
                name=source.replace('Fraction_', ''),
                marker_color=color
            ))

        fig.update_layout(
            barmode='stack',
            title=f'Percentage Contribution of Water Sources to Tree Water - {year}',
            xaxis_title='Month and Tree Type',
            yaxis_title='Percentage (%)',
            legend_title='Source',
            height=500,
            width=800,
            xaxis=dict(
                tickmode='array',
                tickvals=[f'{m} (Fir)' for m in months] + [f'{m} (Spruce)' for m in months],
                ticktext=[f'{m} (Fir)' for m in months] + [f'{m} (Spruce)' for m in months]
            )
        )
        fig.show()

# Print results for inspection
print("Mixing Model Results with Sensitivity Analysis:")
print(results_df)

/var/folders/73/1hl1pq915rg4htnkk3r29mym0000gn/T/ipykernel_99933/3341744009.py:19: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Mixing Model Results with Sensitivity Analysis:
    Year      Month    Type  Fraction_Current_Precip  Fraction_Prev_Precip  \
0   2021       June     Fir                 0.179386              0.820614   
1   2021       June  Spruce                 0.000000              0.859658   
2   2021       July     Fir                 0.000000              0.905528   
3   2021       July  Spruce                 0.000000              0.858406   
4   2021     August     Fir                 0.907429              0.092571   
5   2021     August  Spruce                 0.986817              0.000000   
6   2022       June     Fir                 0.000000              0.000000   
7   2022       June  Spruce                 0.000000              0.000000   
8   2022       July     Fir                 0.000000              0.834811   
9   2022       July  Spruce                 0.000000              0.666078   
10  2022     August     Fir                 0.000000              0.399030   
11  2022     Aug